In [1]:
import os
from openai import OpenAI
from IPython.display import Markdown, display
from bs4 import BeautifulSoup
import requests

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

In [3]:
client = OpenAI(api_key=openai_api_key)

In [4]:
def fetch_website_contents(url):
    """
    Return the title and contents of the website at the given url;
    truncate to 2,000 characters as a sensible limit
    """
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    title = soup.title.string if soup.title else "No title found"
    if soup.body:
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        text = soup.body.get_text(separator="\n", strip=True)
    else:
        text = ""
    return (title + "\n\n" + text)[:2_000]

In [5]:
# ed = fetch_website_contents("https://cnn.com/")
# print(ed)

In [6]:
system_prompt = """
You are a an assistant that analyzes the contents of a website,
and provides a short summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

user_prompt_prefix = """
Here are the contents of a website.
Provide a detailed summary of this website.
If it includes news or announcements, then summarize these too.

"""

def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]

def summarize(url):
    website = fetch_website_contents(url)
    response =  client.chat.completions.create(
        model = "gpt-4.1-mini",
        messages = messages_for(website)
    )
    return response.choices[0].message.content

def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

display_summary("https://www.cnn.com/")

The website is the main page of CNN, a major news organization, offering comprehensive coverage of breaking news, latest news, and video content across a wide array of topics and regions.

### Key Features and Content:
- **News Categories:** The site covers multiple categories including US, World, Politics, Business, Health, Entertainment, Style, Travel, Sports, Science, Climate, and Weather.
- **Major Current Events:** Special emphasis on significant ongoing conflicts such as the Ukraine-Russia War and Israel-Hamas War.
- **Regional Coverage:** Extensive global news segmented by continents (Africa, Americas, Asia, Australia, China, Europe, India, Middle East, United Kingdom).
- **Politics Focus:** Includes US Politics updates, Elections 2026, coverage of prominent political figures such as Trump, fact-checking via "Facts First," polls, and investigative sections like the Epstein Files.
- **Business Section:** Offers news on tech, media, markets (pre-market, after-hours), investing tips, and calculators.
- **Health and Lifestyle:** Features sections on fitness, food, sleep, mindfulness, and relationships.
- **Entertainment:** Covers movies, television, celebrities, as well as tech innovations.
- **Style and Culture:** Includes arts, design, fashion, architecture, luxury, and beauty.
- **Sports:** Comprehensive sports coverage including football, tennis, golf, motorsports, and US sports plus Olympics.
- **Science and Environment:** Includes space exploration, climate change, environment solutions, and weather updates.
- **Media Formats:** Supports articles, videos, live TV streaming, and podcasts or listen options.
- **User Features:** Account sign-in for personalized content, newsletters, topic following, and live TV access.
- **Ad Feedback:** The website actively seeks user feedback on advertisements and technical issues to improve user experience.
- **Languages/ Editions:** Global edition support including US, International, Arabic, and Español.

### Announcements or News:
- There are no specific new announcements or breaking news headlines directly visible in the provided content snippet.
- The site emphasizes ongoing conflicts and major global events implying continuous coverage and updates in those areas.

### User Experience Hindrances Addressed:
- The site provides forms for reporting ad relevance, technical issues with videos or ads, and other user experience feedback.

In summary, CNN’s website serves as a comprehensive portal for up-to-the-minute news across many specialties and regions, enriched with multimedia content and interactive user features designed for a diverse global audience.